In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()

    # --------------------------------------------------
    # 1. Customer ID and duplicate-record handling
    # --------------------------------------------------
    df_clean["customer_id"] = (
        df_clean["customer_id"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    valid_id = df_clean["customer_id"].notna()
    duplicate_mask = valid_id & df_clean.duplicated(subset="customer_id", keep="first")

    duplicate_records_removed = int(duplicate_mask.sum())
    missing_customer_ids = int(df_clean["customer_id"].isna().sum())

    # Retain the first record for each identifiable customer.
    # Records without an ID are retained because they cannot be matched reliably.
    df_clean = df_clean.loc[~duplicate_mask].copy()

    # --------------------------------------------------
    # 2. Country standardization
    # --------------------------------------------------
    countries_map = {
        "Uk": "United Kingdom",
        "Us": "United States",
        "Usa": "United States",
        "U.S.": "United States",
        "Fr": "France",
        "Can": "Canada",
        "Au": "Australia",
        "De": "Germany",
        "Great Britain": "United Kingdom"
    }

    df_clean["country"] = (
        df_clean["country"]
        .astype("string")
        .str.strip()
        .str.title()
        .replace(countries_map)
    )

    # --------------------------------------------------
    # 3. Monthly spend: standardization, anomaly handling
    #    and hierarchical median imputation
    # --------------------------------------------------
    spend_raw = (
        df_clean["monthly_spend"]
        .astype("string")
        .str.strip()
        .str.replace("$", "", regex=False)
    )

    df_clean["monthly_spend"] = pd.to_numeric(spend_raw, errors="coerce")

    df_clean["plan"] = (
        df_clean["plan"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    # Flag negative spend values or the observed 9999 placeholder
    negative_spend_mask = df_clean["monthly_spend"].lt(0).fillna(False)
    placeholder_mask = df_clean["monthly_spend"].eq(9999).fillna(False)
    invalid_spend_base = negative_spend_mask | placeholder_mask

    # Calculate the reference median excluding known invalid/placeholder values
    reference_median = df_clean.loc[~invalid_spend_base, "monthly_spend"].median()

    # Flag unusually high spend values using the 3x reference-median rule
    high_spend_mask = pd.Series(False, index=df_clean.index)
    if pd.notna(reference_median) and reference_median > 0:
        high_spend_mask = (
            ~invalid_spend_base
            & df_clean["monthly_spend"].gt(3 * reference_median)
        ).fillna(False)

    spend_anomaly_mask = invalid_spend_base | high_spend_mask
    df_clean.loc[spend_anomaly_mask, "monthly_spend"] = np.nan

    spend_records_requiring_imputation = int(df_clean["monthly_spend"].isna().sum())

    # Median imputation hierarchy: (Plan + Country) -> (Plan) -> (Global)
    group_spend_median = df_clean.groupby(["plan", "country"])["monthly_spend"].transform("median")
    plan_spend_median = df_clean.groupby("plan")["monthly_spend"].transform("median")
    global_spend_median = df_clean["monthly_spend"].median()

    df_clean["monthly_spend"] = (
        df_clean["monthly_spend"]
        .fillna(group_spend_median)
        .fillna(plan_spend_median)
        .fillna(global_spend_median)
    )

    spend_values_still_missing = int(df_clean["monthly_spend"].isna().sum())

    # --------------------------------------------------
    # 4. Date standardization and parsing
    # --------------------------------------------------
    date_columns = ["last_active_date", "signup_date"]
    date_format_audit = {}

    for col in date_columns:
        raw = df_clean[col].astype("string").str.strip()

        parts = raw.str.extract(r"^(\d{1,2})/(\d{1,2})/(\d{4})$")
        first = pd.to_numeric(parts[0], errors="coerce")
        second = pd.to_numeric(parts[1], errors="coerce")

        dmy_evidence = int(first.gt(12).sum())
        mdy_evidence = int(second.gt(12).sum())
        ambiguous_slash_dates = int((first.between(1, 12) & second.between(1, 12)).sum())

        # Supported formats: YYYY-MM-DD, DD/MM/YYYY (default for ambiguous), MM/DD/YYYY
        date_iso = pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
        date_dmy = pd.to_datetime(raw, format="%d/%m/%Y", errors="coerce")
        date_mdy = pd.to_datetime(raw, format="%m/%d/%Y", errors="coerce")

        parsed_dates = date_iso.fillna(date_dmy).fillna(date_mdy)

        input_present = (raw.notna() & raw.ne("")).fillna(False)
        unparsed_nonempty = int((input_present & parsed_dates.isna()).sum())

        date_format_audit[col] = {
            "DD/MM/YYYY evidence": dmy_evidence,
            "MM/DD/YYYY evidence": mdy_evidence,
            "Ambiguous slash-formatted dates": ambiguous_slash_dates,
            "Non-empty values not parsed": unparsed_nonempty,
            "Missing input values": int((~input_present).sum()),
        }

        df_clean[col] = parsed_dates

    # --------------------------------------------------
    # 5. Churn status and cancellation-reason standardization
    # --------------------------------------------------
    churn_map = {
        "yes": 1, "y": 1, "1": 1, "1.0": 1,
        "no": 0, "n": 0, "0": 0, "0.0": 0,
    }

    churn_raw = df_clean["churned"].astype("string").str.strip().str.lower()
    df_clean["churned"] = churn_raw.map(churn_map).astype("Int64")

    cancellation_raw = (
        df_clean["cancellation_reason"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    df_clean["cancellation_reason"] = cancellation_raw.fillna("Unknown")

    unknown_mask = df_clean["churned"].isna()
    active_mask = df_clean["churned"].eq(0).fillna(False)

    df_clean.loc[active_mask, "cancellation_reason"] = "Not applicable"
    df_clean.loc[unknown_mask, "cancellation_reason"] = (
        df_clean.loc[unknown_mask, "cancellation_reason"]
        .replace("Unknown", "Unknown status")
    )

    # --------------------------------------------------
    # 6. Age and count-feature validation
    # --------------------------------------------------
    df_clean["age"] = pd.to_numeric(df_clean["age"], errors="coerce")

    underage_mask = df_clean["age"].lt(3).fillna(False)
    med_age = df_clean.loc[~underage_mask, "age"].median()

    extreme_ages = pd.Series(False, index=df_clean.index)
    if pd.notna(med_age) and med_age > 0:
        # Based on initial profiling, values > 75 in this dataset correspond to 150 and 999
        extreme_ages = (~underage_mask & df_clean["age"].gt(75)).fillna(False)
    else:
        print("Warning: Unable to calculate extreme age threshold.")

    age_anomaly = underage_mask | extreme_ages
    df_clean.loc[age_anomaly, "age"] = np.nan

    # Hierarchical median imputation for age
    group_age_median = df_clean.groupby(["plan", "country"])["age"].transform("median")
    plan_age_median = df_clean.groupby("plan")["age"].transform("median")
    global_age_median = df_clean["age"].median()

    df_clean["age"] = (
        df_clean["age"]
        .fillna(group_age_median)
        .fillna(plan_age_median)
        .fillna(global_age_median)
    )

    age_values_still_missing = int(df_clean["age"].isna().sum())

    # Ensure count columns are numeric and free of impossible negative values
    for count_col in ["support_tickets", "sessions_last_30d"]:
        if count_col in df_clean.columns:
            df_clean[count_col] = pd.to_numeric(df_clean[count_col], errors="coerce")
            df_clean.loc[df_clean[count_col] < 0, count_col] = np.nan

    # --------------------------------------------------
    # 7. Data-quality audit report
    # --------------------------------------------------
    print("=" * 60)
    print("DATA QUALITY AUDIT REPORT (POST-CLEANING)")
    print("=" * 60)

    print(f"Customer records retained         : {len(df_clean):,}")
    print(f"Duplicate records removed         : {duplicate_records_removed:,}")
    print(f"Records with missing customer IDs : {missing_customer_ids:,}")
    print(f"Underage anomalies (< 3 yrs)      : {int(underage_mask.sum()):,}")
    print(f"Extreme age anomalies (> 75 yrs)  : {int(extreme_ages.sum()):,}")
    print(f"Total age anomalies imputed       : {int(age_anomaly.sum()):,}")
    print(f"Missing ages after imputation     : {age_values_still_missing:,}")

    print("\n--- DataFrame Structure ---")
    df_clean.info()

    print("\n--- Standardized Country Categories ---")
    print(df_clean["country"].dropna().unique().tolist())

    print("\n--- Churn Status Distribution ---")
    churn_labels = (
        df_clean["churned"]
        .map({0: "Active", 1: "Churned"})
        .astype("string")
        .fillna("Unknown status")
    )
    print(churn_labels.value_counts(dropna=False).to_string())

    print("\n--- Top 5 Highest Monthly Spend Values (Post-Cleaning) ---")
    print(df_clean["monthly_spend"].nlargest(5).to_string())

    print("\n--- Monthly Spend Anomaly Audit ---")
    print(f"9999 placeholder records          : {int(placeholder_mask.sum()):,}")
    print(f"Negative spend records            : {int(negative_spend_mask.sum()):,}")
    print(f"Records > 3x reference median     : {int(high_spend_mask.sum()):,}")
    print(f"Total spend anomalies flagged     : {int(spend_anomaly_mask.sum()):,}")
    print(f"Records requiring imputation      : {spend_records_requiring_imputation:,}")
    print(f"Spend values still missing        : {spend_values_still_missing:,}")

    print("\n--- Date Parsing Audit ---")
    for col, audit in date_format_audit.items():
        print(f"\n[{col}]")
        for metric, count in audit.items():
            print(f"  {metric:<33}: {count:,}")

    print("=" * 60 + "\n")
    return df_clean

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency


def _format_p_value(p: float) -> str:
    return "< 0.0001" if p < 0.0001 else f"= {p:.4f}"


def churn_association(
    df: pd.DataFrame,
    x: str,
    churn: str = "churned",
    alpha: float = 0.05,
) -> dict | None:
    # 1. Prepare valid data
    data = df[[x, churn]].dropna(subset=[churn]).copy()
    data = data[data[churn].isin([0, 1])]

    table = pd.crosstab(data[x], data[churn]).reindex(columns=[0, 1], fill_value=0)
    n = int(table.to_numpy().sum())

    if n == 0 or table.shape[0] < 2:
        print(f"Insufficient data or categories for '{x}' to run chi-square test.")
        return None

    # 2. Chi-square test of independence
    chi2, p, dof, expected = chi2_contingency(table, correction=False)

    if dof == 0 or chi2 == 0:
        print(f"No variation observed across categories for '{x}' (chi2 = 0.00).")
        return {
            "chi2": float(chi2),
            "p_value": float(p),
            "dof": int(dof),
            "w": 0.0,
            "significant": False,
            "strength": "None",
        }

    # 3. Cohen's w effect size
    w = float(np.sqrt(chi2 / n))

    # 4. Statistical significance & practical strength (Cohen's w benchmarks)
    significant = bool(p < alpha)

    if w <= 0.10:
        strength = "Weak"
    elif w <= 0.30:
        strength = "Moderate"
    else:
        strength = "Strong"

    p_str = _format_p_value(p)
    print(f"Chi2: {chi2:.2f} (dof={dof}), p {p_str}, Cohen's w: {w:.3f}")

    if expected.min() < 5:
        print("Note: At least one expected cell count is < 5; chi-square approximation may be less accurate.")

    print("\n---- Interpretation ----")
    if significant:
        if strength == "Weak":
            print(
                f"There is a statistically significant, yet weak, association between "
                f"'{churn}' and '{x}' (w = {w:.2f}), indicating a small practical effect."
            )
        elif strength == "Moderate":
            print(
                f"The results reveal a statistically significant and moderate association "
                f"between '{churn}' and '{x}' (w = {w:.2f}), suggesting a meaningful practical relationship."
            )
        else:
            print(
                f"There is a statistically significant and strong association between "
                f"'{churn}' and '{x}' (w = {w:.2f}), demonstrating a substantial practical impact."
            )
    else:
        print(
            f"There is not enough evidence at alpha = {alpha} to reject the "
            f"null hypothesis of independence between '{churn}' and '{x}'."
        )


def data_info(df: pd.DataFrame) -> None:
    df = df.copy()
    sns.set_theme(style="whitegrid")

    # --------------------------------------------------
    # 1. Estimated MRR analysis
    # --------------------------------------------------
    active_customers = df.loc[df["churned"].eq(0)].dropna(subset=["monthly_spend"])
    estimated_mrr = active_customers["monthly_spend"].sum()

    rev_by_plan = (
        active_customers
        .groupby("plan", dropna=False)["monthly_spend"]
        .sum()
        .sort_values(ascending=False)
    )

    rev_by_country = (
        active_customers
        .groupby("country", dropna=False)["monthly_spend"]
        .sum()
        .sort_values(ascending=False)
    )

    print("=" * 60)
    print("1. ESTIMATED MONTHLY RECURRING REVENUE (MRR)")
    print("=" * 60)
    print(f"Total Estimated MRR (Active Customers): ${estimated_mrr:,.2f}\n")

    print("--- Estimated MRR by Subscription Plan ---")
    print(rev_by_plan.map("${:,.2f}".format).to_string())

    print("\n--- Estimated MRR by Customer Country ---")
    print(rev_by_country.map("${:,.2f}".format).to_string())

    rev_matrix = (
        active_customers
        .groupby(["plan", "country"], dropna=False)["monthly_spend"]
        .sum()
        .unstack(fill_value=0)
    )

    if estimated_mrr > 0:
        perc_matrix = (rev_matrix / estimated_mrr) * 100
        print("\n--- Percentage Contribution to Estimated MRR (Plan x Country %) ---")
        print(perc_matrix.round(2).to_string())

        # Visualization: Clean horizontal bar charts (easier to compare than pie charts)
        rev_by_plan_plot = rev_by_plan.rename(index=lambda x: "Unknown" if pd.isna(x) else str(x))
        rev_by_country_plot = rev_by_country.rename(index=lambda x: "Unknown" if pd.isna(x) else str(x))

        fig, ax = plt.subplots(1, 2, figsize=(14, 5))

        sns.barplot(
            x=rev_by_plan_plot.values,
            y=rev_by_plan_plot.index,
            hue=rev_by_plan_plot.index,
            palette="Blues_d",
            legend=False,
            ax=ax[0],
        )
        ax[0].set_title("Estimated MRR by Subscription Plan", fontsize=12, fontweight="bold")
        ax[0].set_xlabel("Estimated MRR ($)")
        ax[0].set_ylabel("")
        for i, v in enumerate(rev_by_plan_plot.values):
            pct = (v / estimated_mrr) * 100
            ax[0].text(v, i, f" ${v:,.0f} ({pct:.1f}%)", va="center", fontsize=9)
        ax[0].set_xlim(0, rev_by_plan_plot.max() * 1.25)

        sns.barplot(
            x=rev_by_country_plot.values,
            y=rev_by_country_plot.index,
            hue=rev_by_country_plot.index,
            palette="Teal_d" if "Teal_d" in sns.palettes.SEABORN_PALETTES else "crest",
            legend=False,
            ax=ax[1],
        )
        ax[1].set_title("Estimated MRR by Customer Country", fontsize=12, fontweight="bold")
        ax[1].set_xlabel("Estimated MRR ($)")
        ax[1].set_ylabel("")
        for i, v in enumerate(rev_by_country_plot.values):
            pct = (v / estimated_mrr) * 100
            ax[1].text(v, i, f" ${v:,.0f} ({pct:.1f}%)", va="center", fontsize=9)
        ax[1].set_xlim(0, rev_by_country_plot.max() * 1.25)

        sns.despine()
        plt.tight_layout()
        plt.savefig("fig1_mrr.png",dpi=300,bbox_inches="tight")
        plt.show()
    else:
        print("\nMRR breakdown charts skipped: Estimated MRR is zero or no eligible records exist.")

    # --------------------------------------------------
    # 2. Customer duration and support activity analysis
    # --------------------------------------------------
    df["tenure_days"] = (df["last_active_date"] - df["signup_date"]).dt.days
    df.loc[df["tenure_days"] < 0, "tenure_days"] = np.nan

    df_tenure = df.dropna(subset=["tenure_days"])
    clean_tenure = df_tenure["tenure_days"].to_numpy()

    if len(clean_tenure) > 0:
        q50, q5, q95 = np.percentile(clean_tenure, [50, 5, 95])
    else:
        q50 = q5 = q95 = np.nan

    churned_df = df.loc[df["churned"].eq(1)].copy()
    mean_observed_duration_churned = churned_df["tenure_days"].mean()

    cus_with_tick = df_tenure.loc[df_tenure["support_tickets"] > 0].copy()
    cus_with_tick["days_per_ticket"] = cus_with_tick["tenure_days"] / cus_with_tick["support_tickets"]
    median_days_per_ticket = cus_with_tick["days_per_ticket"].median()

    print("\n" + "=" * 60)
    print("2. CUSTOMER DURATION & SUPPORT ACTIVITY")
    print("=" * 60)
    print(f"Median signup-to-last-activity duration : {q50:.2f} days (IQR 5th–95th: {q5:.2f} to {q95:.2f} days)")
    print(f"Mean duration (confirmed churned)       : {mean_observed_duration_churned:.2f} days")
    print(f"Median active days per support ticket   : {median_days_per_ticket:.2f} days/ticket")

    # --------------------------------------------------
    # 3. Customer churn profile & statistical association
    # --------------------------------------------------
    df["churn_status"] = (
        df["churned"]
        .map({0: "Active", 1: "Churned"})
        .astype("string")
        .fillna("Unknown status")
    )

    profile_columns = ["monthly_spend", "sessions_last_30d", "support_tickets", "tenure_days"]
    advanced_churn_profile = df.groupby("churn_status")[profile_columns].median()

    print("\n" + "=" * 60)
    print("3. CUSTOMER PROFILE BY CHURN STATUS (MEDIAN VALUES)")
    print("=" * 60)
    print(advanced_churn_profile.round(2).to_string())

    print("\n" + "=" * 60)
    print("4. CHURN ASSOCIATION ANALYSIS (CHI-SQUARE & EFFECT SIZE)")
    print("=" * 60)

    data = df[df["churned"].isin([0, 1])].copy()

    # Helper to compute and print contingency summaries consistently
    def _summarize_segment(segment_col: str, title: str, bin_description: str) -> None:
        print(f"\n--- {title} ---")
        table = pd.crosstab(data[segment_col], data["churned"]).reindex(columns=[0, 1], fill_value=0)
        summary = pd.DataFrame({
            "Total Customers": table.sum(axis=1),
            "Churned": table[1],
            "Churn Rate (%)": (table[1] / table.sum(axis=1) * 100).round(2),
        })
        print(summary.to_string())
        print(bin_description)
        print(f"\nStatistical Association ({title}):")
        churn_association(data, x=segment_col)

    # A. Age vs. Churn
    q1_age, q3_age = data["age"].quantile([0.25, 0.75])
    data["age_group"] = pd.cut(
        data["age"],
        bins=[-np.inf, q1_age, q3_age, np.inf],
        labels=["Low age", "Middle age", "High age"],
        include_lowest=True,
    )
    _summarize_segment(
        segment_col="age_group",
        title="Churn vs. Age Group",
        bin_description=(
            f"  * Low age   : <= {q1_age:.1f} years\n"
            f"  * Middle age: > {q1_age:.1f} and <= {q3_age:.1f} years\n"
            f"  * High age  : > {q3_age:.1f} years"
        ),
    )

    # B. Monthly Spend vs. Churn
    q1_spend, q3_spend = data["monthly_spend"].quantile([0.25, 0.75])
    data["spend_group"] = pd.cut(
        data["monthly_spend"],
        bins=[-np.inf, q1_spend, q3_spend, np.inf],
        labels=["Low spend", "Middle spend", "High spend"],
        include_lowest=True,
    )
    _summarize_segment(
        segment_col="spend_group",
        title="Churn vs. Monthly Spend",
        bin_description=(
            f"  * Low spend   : <= ${q1_spend:.2f}\n"
            f"  * Middle spend: > ${q1_spend:.2f} and <= ${q3_spend:.2f}\n"
            f"  * High spend  : > ${q3_spend:.2f}"
        ),
    )

    # C. Support Tickets vs. Churn
    data["ticket_group"] = pd.cut(
        data["support_tickets"],
        bins=[-np.inf, 0, 2, np.inf],
        labels=["0 tickets", "1-2 tickets", "3+ tickets"],
        include_lowest=True,
    )
    _summarize_segment(
        segment_col="ticket_group",
        title="Churn vs. Support Tickets",
        bin_description=(
            "  * 0 tickets  : 0 support tickets\n"
            "  * 1-2 tickets: 1 to 2 support tickets\n"
            "  * 3+ tickets : 3 or more support tickets"
        ),
    )

    # --------------------------------------------------
    # 7. Churn Rate Comparison Across Key Segments
    # --------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    overall_churn = data["churned"].mean() * 100

    segments = [
        ("ticket_group", "Churn Rate by Support Tickets", "#c0392b"),
        ("spend_group", "Churn Rate by Monthly Spend", "#2980b9"),
        ("age_group", "Churn Rate by Age Group", "#16a085"),
    ]

    for ax, (col, title, color) in zip(axes, segments):
        rates = (
            data.groupby(col, observed=False)["churned"]
            .mean()
            .mul(100)
        )
        bars = ax.bar(rates.index.astype(str), rates.values, color=color, alpha=0.85, width=0.55)
        ax.axhline(overall_churn, color="#333333", linestyle="--", linewidth=1.2, label=f"Avg: {overall_churn:.1f}%")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_ylim(0, 48)
        ax.tick_params(axis="x", rotation=15)
        ax.legend(loc="upper right", fontsize=8.5)

        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 1.0, f"{h:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

    axes[0].set_ylabel("Churn Rate (%)")
    sns.despine()
    plt.tight_layout()
    plt.savefig("churn_drivers_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()

   # --------------------------------------------------
    # 5. Cancellation-reason analysis & visualization
    # --------------------------------------------------
    cancellation_counts = churned_df["cancellation_reason"].value_counts(dropna=False)
    cancellation_breakdown = (
        churned_df["cancellation_reason"]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
    )

    print("\n" + "=" * 60)
    print("5. CANCELLATION REASON BREAKDOWN (CONFIRMED CHURNED %)")
    print("=" * 60)
    if len(cancellation_breakdown) > 0:
        print(
            cancellation_breakdown
            .astype(float)
            .round(2)
            .map("{:.2f}%".format)
            .to_string()
        )

        fig, ax = plt.subplots(figsize=(9, 4.5))

        reasons = cancellation_breakdown.index.astype(str).tolist()
        pcts = cancellation_breakdown.to_numpy(dtype=float)
        counts = cancellation_counts.to_numpy(dtype=int)

        sns.barplot(
            x=pcts,
            y=reasons,
            hue=reasons,
            palette="Reds_r",
            legend=False,
            ax=ax,
        )

        ax.set_title(
            f"Cancellation Reasons Among Churned Subscribers (n = {len(churned_df):,})",
            fontsize=12.5,
            fontweight="bold",
        )
        ax.set_xlabel("Share of Churned Customers (%)")
        ax.set_ylabel("")
        ax.set_xlim(0, float(np.max(pcts)) * 1.28)

        for i, (pct, cnt) in enumerate(zip(pcts, counts)):
            ax.text(
                float(pct) + 0.4,
                i,
                f"{float(pct):.2f}% ({int(cnt):,})",
                va="center",
                fontsize=9.5,
                fontweight="bold",
                color="#333333",
            )

        sns.despine()
        plt.tight_layout()
        plt.savefig("cancellation_reasons.png", dpi=300, bbox_inches="tight")
        plt.show()
    else:
        print("No confirmed churned records available for cancellation-reason analysis.")
    # --------------------------------------------------
    # 6. Age distribution visualization
    # --------------------------------------------------
    age = df["age"].dropna()
    q1, q3 = age.quantile([0.25, 0.75])

    fig, ax = plt.subplots(figsize=(9, 4.8))
    sns.kdeplot(age, fill=True, color="#2b5c8f", alpha=0.35, linewidth=2, ax=ax)
    ax.axvline(q1, color="#d95f02", linestyle="--", linewidth=1.5, label=f"Q1 (25th pct) = {q1:.1f} yrs")
    ax.axvline(q3, color="#1b9e77", linestyle="--", linewidth=1.5, label=f"Q3 (75th pct) = {q3:.1f} yrs")

    y_max = ax.get_ylim()[1]
    ax.axvspan(q1, q3, color="#2b5c8f", alpha=0.08)
    ax.text(
        (q1 + q3) / 2,
        y_max * 0.82,
        "Core 50% (IQR)\nPrimary customer age range",
        ha="center",
        va="center",
        fontsize=9.5,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#cccccc", alpha=0.9),
    )

    ax.set_title("Customer Age Distribution", fontsize=13, fontweight="bold")
    ax.set_xlabel("Age (Years)")
    ax.set_ylabel("Density")
    ax.legend(frameon=True)
    sns.despine()
    plt.tight_layout()
    plt.savefig("fig4_agedist.png",dpi=300,bbox_inches="tight")
    plt.show()